# Orbit Viewer — 99942 Apophis
Visualizador de planos orbitales inspirado en el **JPL SBDB View Orbit Plane (VOP)**, construido sobre `pymcel` y `Plotly`.

Muestra las órbitas de Mercurio, Venus, Tierra, Marte y **Apophis**, con:
- **Tick marks** (picket-fence) que ilustran la inclinación respecto a la eclíptica.
- **Posición actual** de cada cuerpo al **2026-04-22 00:00 UTC**.
- **Línea de nodos** (amarilla) y **normal al plano orbital** (verde) de Apophis.
- Banda sombreada entre la eclíptica y el plano orbital de Apophis.

In [ ]:
# Instalar dependencias (omitir si ya están instaladas)
import sys
!{sys.executable} -m pip install -q pymcel plotly numpy

## Imports

In [ ]:
import numpy as np
from orbit_viewer import (
    OrbitElements, sample_times, propagate_two_body, make_orbit_viewer
)

## Constantes y elementos orbitales (J2000)

In [ ]:
# Parámetro gravitacional del Sol  [AU³/día²]
MU_SUN = 2.9591220828559093e-4

# Días desde J2000 (2000-Jan-1.5) hasta 2026-Apr-22.0
T_CURRENT = 9608.5

# ─── Elementos orbitales en J2000  (a [AU], ángulos en grados, M0 en grados) ───
MERCURY = OrbitElements(a=0.38710, e=0.20563, i=7.005,  raan=48.331,  argp=29.124,  M0=174.796, epoch=0.0, mu=MU_SUN)
VENUS   = OrbitElements(a=0.72333, e=0.00677, i=3.395,  raan=76.680,  argp=55.186,  M0=50.114,  epoch=0.0, mu=MU_SUN)
EARTH   = OrbitElements(a=1.00000, e=0.01671, i=0.000,  raan=-11.261, argp=102.937, M0=8.789,   epoch=0.0, mu=MU_SUN)
MARS    = OrbitElements(a=1.52366, e=0.09341, i=1.850,  raan=49.562,  argp=-73.517, M0=19.402,  epoch=0.0, mu=MU_SUN)
APOPHIS = OrbitElements(a=0.92238, e=0.19117, i=3.341,  raan=203.900, argp=126.673, M0=122.0,   epoch=0.0, mu=MU_SUN)

## Propagación de órbitas

In [ ]:
def propagate_body(elements, n_pts=500):
    """Propaga un cuerpo por una órbita completa y devuelve trayectoria + posición actual."""
    T = 2 * np.pi * np.sqrt(elements.a**3 / elements.mu)  # período [días]
    times = sample_times(0.0, T, n_pts)
    positions = propagate_two_body(elements, times)
    epoch_pos = propagate_two_body(elements, np.array([T_CURRENT]))[0]
    return positions, epoch_pos

print('Propagando órbitas...')
mercury_pos, mercury_now = propagate_body(MERCURY)
venus_pos,   venus_now   = propagate_body(VENUS)
earth_pos,   earth_now   = propagate_body(EARTH)
mars_pos,    mars_now    = propagate_body(MARS)
apophis_pos, apophis_now = propagate_body(APOPHIS)
print('¡Listo!')

## Visualización interactiva
La figura es un `plotly.graph_objects.Figure` embebido en el notebook.  
Usa los controles de Plotly para rotar, hacer zoom y activar/desactivar trazas.

In [ ]:
reference_bodies = [
    {"name": "Mercurio", "positions": mercury_pos, "color": "#ff80c0", "epoch_pos": mercury_now},
    {"name": "Venus",    "positions": venus_pos,   "color": "#9040d0", "epoch_pos": venus_now},
    {"name": "Tierra",   "positions": earth_pos,   "color": "#40a0ff", "epoch_pos": earth_now},
    {"name": "Marte",    "positions": mars_pos,    "color": "#ff5500", "epoch_pos": mars_now},
]

fig = make_orbit_viewer(
    body_name="99942 Apophis",
    body_positions=apophis_pos,
    body_elements=APOPHIS,
    body_color="#e0e0e0",
    body_epoch_pos=apophis_now,
    reference_bodies=reference_bodies,
    epoch_label="2026-04-22 00:00:00 UTC",
    title="Orbit Viewer — 99942 Apophis",
    n_ticks=90,
    show_wedge=True,
)
fig.show()